# 00 · Build the dataset: raw ParlaSpeech-HR → cleaned, speaker- and session-disjoint splits

Runs on Kaggle. CPU is enough. **Settings → Internet: On.** No manual uploads.

Optional: a Kaggle secret `HF_TOKEN` (Hugging Face read token) avoids anonymous rate limits.

Output: `/kaggle/working/dataset_release.zip` with `data/raw`, `data/clean`, `data/audio`.
Unzip it at the repository root and commit it.

In [ ]:
REPO_URL = "https://github.com/eneserman/NLP.git"  # the public repository
BRANCH = "main"

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret; using anonymous access")

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -b $BRANCH $REPO_URL repo
%cd /kaggle/working/repo
!pip install -q -r requirements/data.txt
!ffmpeg -hide_banner -encoders | grep libopus

Tests of the data pipeline and the evaluation code.

In [ ]:
!python -m pytest -q

## Step 1 · Metadata of the full corpus (no audio); records the source commit SHA

In [ ]:
!python scripts/01_fetch_metadata.py --workers 16

## Step 2 · Speakers of ParlaSpeech-HR v1.0 (excluded from dev/test)

In [ ]:
!python scripts/02_fetch_v1_speakers.py

## Step 3 · Choose speakers, sessions and clips

In [ ]:
!python scripts/03_select_subset.py

## Step 4 · Download the chosen clips → 16 kHz mono Opus; raw manifest

In [ ]:
!python scripts/04_fetch_audio.py --workers 8

## Step 5 · Cleaning (every removed clip is logged with its reason)

In [ ]:
!python scripts/05_clean.py --workers 8

## Step 6 · Speaker- and session-disjoint split; statistics

In [ ]:
!python scripts/06_split.py

## Step 7 · No-speech test clips (W9 slide 44)

In [ ]:
!python scripts/07_make_silence_probe.py

## Step 8 · Pack audio into shards under 90 MB

In [ ]:
!python scripts/08_pack_audio.py && du -sh data/*

In [ ]:
!cd /kaggle/working/repo && zip -q -r /kaggle/working/dataset_release.zip data && ls -lh /kaggle/working/dataset_release.zip